# 🎯 PRÁCTICA: Regresión Logística y Boosting con Iris + MLflow

En esta práctica vamos a:
1. Cargar el dataset Iris
2. Entrenar dos modelos: Regresión Logística y Gradient Boosting
3. Comparar performance
4. **Guardar ambos modelos en MLflow**
5. Registrar métricas y parámetros

---

## 1️⃣ Importar librerías


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)
import mlflow
import mlflow.sklearn
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías importadas")


✅ Librerías importadas


## 2️⃣ Configurar MLflow


In [ ]:
# Conectar a MLflow (asegurate que http://localhost:5000 esté disponible)
mlflow.set_tracking_uri("http://localhost:5000")

# Crear o usar experimento
experiment_name = "Práctica Iris - Regresión Logística vs Boosting"
try:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"✅ Nuevo experimento creado: {experiment_name}")
except:
    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id
    print(f"✅ Usando experimento existente: {experiment_name}")

print(f"📊 Experiment ID: {experiment_id}")
print(f"🌐 MLflow UI: http://localhost:5000")


✅ Nuevo experimento creado: Práctica Iris - Regresión Logística vs Boosting
📊 Experiment ID: 2
🌐 MLflow UI: http://localhost:5000


## 3️⃣ Cargar y preparar datos


In [3]:
# Cargar Iris
iris = load_iris()
X = iris.data
y = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

print(f"📊 Dataset Iris")
print(f"  - Shape: {X.shape}")
print(f"  - Features: {len(feature_names)}")
print(f"  - Clases: {len(target_names)}")
print(f"  - Distribución: {np.bincount(y)}")
print()

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"✅ Split:")
print(f"  - Train: {X_train.shape[0]} muestras")
print(f"  - Test: {X_test.shape[0]} muestras")


📊 Dataset Iris
  - Shape: (150, 4)
  - Features: 4
  - Clases: 3
  - Distribución: [50 50 50]

✅ Split:
  - Train: 120 muestras
  - Test: 30 muestras


## 4️⃣ Modelo 1: Regresión Logística


In [4]:
print("\n" + "="*60)
print("MODELO 1: REGRESIÓN LOGÍSTICA")
print("="*60 + "\n")

# Entrenar modelo
lr_model = LogisticRegression(
    max_iter=200,
    random_state=42,
    multi_class='multinomial'
)
lr_model.fit(X_train, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test)
y_pred_proba_lr = lr_model.predict_proba(X_test)

# Métricas
metrics_lr = {
    'accuracy': accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'f1': f1_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_lr, multi_class='ovr'),
}

print("📊 Métricas Regresión Logística:")
for metric, value in metrics_lr.items():
    print(f"  - {metric:12}: {value:.4f}")

# Validación cruzada
cv_scores_lr = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='f1_weighted')
print(f"\n  - CV (5-fold): {cv_scores_lr.mean():.4f} (+/- {cv_scores_lr.std():.4f})")



MODELO 1: REGRESIÓN LOGÍSTICA

📊 Métricas Regresión Logística:
  - accuracy    : 0.9667
  - precision   : 0.9697
  - recall      : 0.9667
  - f1          : 0.9666
  - roc_auc     : 1.0000

  - CV (5-fold): 0.9665 (+/- 0.0167)


## 5️⃣ Modelo 2: Gradient Boosting


In [5]:
print("\n" + "="*60)
print("MODELO 2: GRADIENT BOOSTING")
print("="*60 + "\n")

# Entrenar modelo
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
gb_model.fit(X_train, y_train)

# Predicciones
y_pred_gb = gb_model.predict(X_test)
y_pred_proba_gb = gb_model.predict_proba(X_test)

# Métricas
metrics_gb = {
    'accuracy': accuracy_score(y_test, y_pred_gb),
    'precision': precision_score(y_test, y_pred_gb, average='weighted', zero_division=0),
    'recall': recall_score(y_test, y_pred_gb, average='weighted', zero_division=0),
    'f1': f1_score(y_test, y_pred_gb, average='weighted', zero_division=0),
    'roc_auc': roc_auc_score(y_test, y_pred_proba_gb, multi_class='ovr'),
}

print("📊 Métricas Gradient Boosting:")
for metric, value in metrics_gb.items():
    print(f"  - {metric:12}: {value:.4f}")

# Validación cruzada
cv_scores_gb = cross_val_score(gb_model, X_train, y_train, cv=5, scoring='f1_weighted')
print(f"\n  - CV (5-fold): {cv_scores_gb.mean():.4f} (+/- {cv_scores_gb.std():.4f})")



MODELO 2: GRADIENT BOOSTING

📊 Métricas Gradient Boosting:
  - accuracy    : 0.9667
  - precision   : 0.9697
  - recall      : 0.9667
  - f1          : 0.9666
  - roc_auc     : 0.9900

  - CV (5-fold): 0.9665 (+/- 0.0167)


## 6️⃣ Comparar modelos


In [7]:
# Comparación en tabla
comparison_df = pd.DataFrame({
    'Regresión Logística': metrics_lr,
    'Gradient Boosting': metrics_gb
}).T

print("\n" + "="*60)
print("COMPARACIÓN DE MODELOS")
print("="*60)
print(comparison_df.round(4))

# Diferencia
print("\n🔍 Diferencia (GB - LR):")
for metric in metrics_lr.keys():
    diff = metrics_gb[metric] - metrics_lr[metric]
    symbol = "📈" if diff > 0 else "📉"
    print(f"  {symbol} {metric:12}: {diff:+.4f}")

# Mejor modelo
mejor_f1 = 'Gradient Boosting' if metrics_gb['f1'] > metrics_lr['f1'] else 'Regresión Logística'
print(f"\n🏆 Mejor modelo (F1): {mejor_f1}")



COMPARACIÓN DE MODELOS
                     accuracy  precision  recall      f1  roc_auc
Regresión Logística    0.9667     0.9697  0.9667  0.9666     1.00
Gradient Boosting      0.9667     0.9697  0.9667  0.9666     0.99

🔍 Diferencia (GB - LR):
  📉 accuracy    : +0.0000
  📉 precision   : +0.0000
  📉 recall      : +0.0000
  📉 f1          : +0.0000
  📉 roc_auc     : -0.0100

🏆 Mejor modelo (F1): Regresión Logística


## 7️⃣ Guardar Modelo 1 en MLflow (Regresión Logística)


In [8]:
print("\n" + "="*60)
print("GUARDANDO EN MLflow: REGRESIÓN LOGÍSTICA")
print("="*60 + "\n")

with mlflow.start_run(experiment_id=experiment_id, run_name="Logistic Regression"):
    # Log parámetros
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 200)
    mlflow.log_param("random_state", 42)
    
    # Log métricas
    for metric_name, metric_value in metrics_lr.items():
        mlflow.log_metric(metric_name, metric_value)
    
    # Log CV scores
    mlflow.log_metric("cv_mean_f1", cv_scores_lr.mean())
    mlflow.log_metric("cv_std_f1", cv_scores_lr.std())
    
    # Log modelo
    mlflow.sklearn.log_model(lr_model, "model")
    
    # Log información adicional
    mlflow.log_param("dataset", "Iris")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    run_id_lr = mlflow.active_run().info.run_id
    print(f"✅ Regresión Logística guardada")
    print(f"   Run ID: {run_id_lr}")
    print(f"   Metrics logged: {len(metrics_lr)}")


2025/11/17 23:49:04 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet




GUARDANDO EN MLflow: REGRESIÓN LOGÍSTICA

✅ Regresión Logística guardada
   Run ID: 0764b3499a274c218c95448b65bf6289
   Metrics logged: 5


## 8️⃣ Guardar Modelo 2 en MLflow (Gradient Boosting)


In [9]:
print("\n" + "="*60)
print("GUARDANDO EN MLflow: GRADIENT BOOSTING")
print("="*60 + "\n")

with mlflow.start_run(experiment_id=experiment_id, run_name="Gradient Boosting"):
    # Log parámetros
    mlflow.log_param("model_type", "GradientBoostingClassifier")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 3)
    mlflow.log_param("random_state", 42)
    
    # Log métricas
    for metric_name, metric_value in metrics_gb.items():
        mlflow.log_metric(metric_name, metric_value)
    
    # Log CV scores
    mlflow.log_metric("cv_mean_f1", cv_scores_gb.mean())
    mlflow.log_metric("cv_std_f1", cv_scores_gb.std())
    
    # Log modelo
    mlflow.sklearn.log_model(gb_model, "model")
    
    # Log información adicional
    mlflow.log_param("dataset", "Iris")
    mlflow.log_param("train_size", len(X_train))
    mlflow.log_param("test_size", len(X_test))
    
    run_id_gb = mlflow.active_run().info.run_id
    print(f"✅ Gradient Boosting guardado")
    print(f"   Run ID: {run_id_gb}")
    print(f"   Metrics logged: {len(metrics_gb)}")



GUARDANDO EN MLflow: GRADIENT BOOSTING

✅ Gradient Boosting guardado
   Run ID: 99f1e4d94fbc429e9248856a6d6ade30
   Metrics logged: 5


## 9️⃣ Visualizar resultados


In [10]:
# Generar matrices de confusión sin visualización gráfica
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_gb = confusion_matrix(y_test, y_pred_gb)

print("\\n📊 Matriz de Confusión - Regresión Logística:")
for row in cm_lr:
    print(f"  {row}")

print("\\n📊 Matriz de Confusión - Gradient Boosting:")
for row in cm_gb:
    print(f"  {row}")

print("\\n✅ Análisis completado (visualizaciones omitidas - librería seaborn no disponible)")


\n📊 Matriz de Confusión - Regresión Logística:
  [10  0  0]
  [0 9 1]
  [ 0  0 10]
\n📊 Matriz de Confusión - Gradient Boosting:
  [10  0  0]
  [0 9 1]
  [ 0  0 10]
\n✅ Análisis completado (visualizaciones omitidas - librería seaborn no disponible)


## 🔟 Resumen y próximos pasos


In [11]:
print("\n" + "="*60)
print("RESUMEN Y ACCIONES")
print("="*60)

print(f"\n📊 Experimento: {experiment_name}")
print(f"🔗 MLflow UI: http://localhost:5000")
print(f"\n📁 Modelos guardados:")
print(f"   1. Regresión Logística (Run ID: {run_id_lr[:8]}...)")
print(f"   2. Gradient Boosting (Run ID: {run_id_gb[:8]}...)")

print(f"\n🏆 Mejor modelo: {mejor_f1}")
print(f"   - F1 Score LR: {metrics_lr['f1']:.4f}")
print(f"   - F1 Score GB: {metrics_gb['f1']:.4f}")
print(f"   - Diferencia: {abs(metrics_gb['f1'] - metrics_lr['f1']):+.4f}")

print(f"\n💡 Próximos pasos:")
print(f"   1. Abre http://localhost:5000 para ver ambos modelos")
print(f"   2. Compara las métricas en la interfaz de MLflow")
print(f"   3. Descarga los modelos para hacer predicciones")
print(f"   4. Experimenta con diferentes hiperparámetros")

print("\n✅ PRÁCTICA COMPLETADA")
print("="*60)



RESUMEN Y ACCIONES

📊 Experimento: Práctica Iris - Regresión Logística vs Boosting
🔗 MLflow UI: http://localhost:5000

📁 Modelos guardados:
   1. Regresión Logística (Run ID: 0764b349...)
   2. Gradient Boosting (Run ID: 99f1e4d9...)

🏆 Mejor modelo: Regresión Logística
   - F1 Score LR: 0.9666
   - F1 Score GB: 0.9666
   - Diferencia: +0.0000

💡 Próximos pasos:
   1. Abre http://localhost:5000 para ver ambos modelos
   2. Compara las métricas en la interfaz de MLflow
   3. Descarga los modelos para hacer predicciones
   4. Experimenta con diferentes hiperparámetros

✅ PRÁCTICA COMPLETADA


---

# PARTE 2: SIMULACIÓN DE DRIFT FUTURO 🚀

## 🎯 Objetivo: 
Simular cómo se vería el modelo con datos del futuro que han cambiado.
Los alumnos deben **identificar qué escenarios tienen drift** analizando los reportes.


## 📋 Escenarios a simular:
1. **Escenario A**: Datos sin cambios (baseline)
2. **Escenario B**: Covariate Shift - features cambian
3. **Escenario C**: Label Shift - proporción de clases cambia
4. **Escenario D**: Feature Drift - outliers y ruido


## 🔄 Escenario A: Sin cambios (Baseline)


In [12]:
import os
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset

print("\n" + "="*60)
print("ESCENARIO A: SIN CAMBIOS (BASELINE)")
print("="*60 + "\n")

# Generar datos sin drift (pequeño ruido normal)
np.random.seed(42)
indices_a = np.random.choice(len(X_test), size=100, replace=True)
X_future_a = X_test[indices_a] + np.random.normal(0, 0.01, X_test[indices_a].shape)
y_future_a = y_test[indices_a]

# Crear DataFrames para Evidently
df_train_ref = pd.DataFrame(X_train, columns=feature_names)
df_train_ref["target"] = y_train
df_train_ref["prediction"] = gb_model.predict(X_train)

df_future_a = pd.DataFrame(X_future_a, columns=feature_names)
df_future_a["target"] = y_future_a
df_future_a["prediction"] = gb_model.predict(X_future_a)

# Generar reporte
report_a = Report(metrics=[DataDriftPreset()])
report_a.run(reference_data=df_train_ref, current_data=df_future_a)

# Guardar HTML
os.makedirs("/app/reports", exist_ok=True)
report_a_path = "/app/reports/practica_escenario_a_sin_cambios.html"
report_a.save_html(report_a_path)

print(f"✅ Reporte guardado: {report_a_path}")
print(f"📊 Muestras futuras generadas: {len(X_future_a)}")
print(f"🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!")



ESCENARIO A: SIN CAMBIOS (BASELINE)

✅ Reporte guardado: /app/reports/practica_escenario_a_sin_cambios.html
📊 Muestras futuras generadas: 100
🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!


## 🔄 Escenario B: Covariate Shift


In [13]:
print("\n" + "="*60)
print("ESCENARIO B: COVARIATE SHIFT (cambio en features)")
print("="*60 + "\n")

# Los features cambian pero las clases se mantienen igual
indices_b = np.random.choice(len(X_test), size=100, replace=True)
X_future_b = X_test[indices_b].copy().astype(float)

# Aplicar shift: aumentar media de todos los features
shift_magnitude = 0.5
for i in range(X_future_b.shape[1]):
    X_future_b[:, i] = X_future_b[:, i] + np.random.normal(shift_magnitude, 0.2, len(X_future_b))

y_future_b = y_test[indices_b]

# Crear DataFrames
df_future_b = pd.DataFrame(X_future_b, columns=feature_names)
df_future_b["target"] = y_future_b
df_future_b["prediction"] = gb_model.predict(X_future_b)

# Generar reporte
report_b = Report(metrics=[DataDriftPreset()])
report_b.run(reference_data=df_train_ref, current_data=df_future_b)

# Guardar HTML
report_b_path = "/app/reports/practica_escenario_b_covariate_shift.html"
report_b.save_html(report_b_path)

print(f"✅ Reporte guardado: {report_b_path}")
print(f"📊 Cambio aplicado: Media de features +{shift_magnitude}")
print(f"🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!")



ESCENARIO B: COVARIATE SHIFT (cambio en features)

✅ Reporte guardado: /app/reports/practica_escenario_b_covariate_shift.html
📊 Cambio aplicado: Media de features +0.5
🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!


## 🔄 Escenario C: Label Shift


In [14]:
print("\n" + "="*60)
print("ESCENARIO C: LABEL SHIFT (cambio en proporción de clases)")
print("="*60 + "\n")

# Cambiar distribución de clases: más Clase 0, menos Clase 2
n_samples = 100
class_0_count = int(0.7 * n_samples)  # 70% clase 0
class_1_count = int(0.2 * n_samples)  # 20% clase 1
class_2_count = n_samples - class_0_count - class_1_count  # 10% clase 2

indices_0 = np.random.choice(np.where(y_test == 0)[0], size=class_0_count, replace=True)
indices_1 = np.random.choice(np.where(y_test == 1)[0], size=class_1_count, replace=True)
indices_2 = np.random.choice(np.where(y_test == 2)[0], size=class_2_count, replace=True)

X_future_c = np.vstack([X_test[indices_0], X_test[indices_1], X_test[indices_2]])
y_future_c = np.concatenate([y_test[indices_0], y_test[indices_1], y_test[indices_2]])

# Crear DataFrames
df_future_c = pd.DataFrame(X_future_c, columns=feature_names)
df_future_c["target"] = y_future_c
df_future_c["prediction"] = gb_model.predict(X_future_c)

# Generar reporte
report_c = Report(metrics=[DataDriftPreset()])
report_c.run(reference_data=df_train_ref, current_data=df_future_c)

# Guardar HTML
report_c_path = "/app/reports/practica_escenario_c_label_shift.html"
report_c.save_html(report_c_path)

print(f"✅ Reporte guardado: {report_c_path}")
print(f"📊 Nueva distribución: Clase 0={class_0_count}%, Clase 1={class_1_count}%, Clase 2={class_2_count}%")
print(f"🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!")



ESCENARIO C: LABEL SHIFT (cambio en proporción de clases)

✅ Reporte guardado: /app/reports/practica_escenario_c_label_shift.html
📊 Nueva distribución: Clase 0=70%, Clase 1=20%, Clase 2=10%
🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!


## 🔄 Escenario D: Feature Drift


In [15]:
print("\n" + "="*60)
print("ESCENARIO D: FEATURE DRIFT (outliers y ruido)")
print("="*60 + "\n")

# Generar datos con outliers y mucho ruido
indices_d = np.random.choice(len(X_test), size=100, replace=True)
X_future_d = X_test[indices_d].copy().astype(float)
y_future_d = y_test[indices_d]

# Agregar outliers (10% de datos)
outlier_indices = np.random.choice(len(X_future_d), size=int(0.1 * len(X_future_d)), replace=False)
for idx in outlier_indices:
    feature_idx = np.random.randint(0, X_future_d.shape[1])
    X_future_d[idx, feature_idx] = np.random.uniform(-3, 3)

# Agregar ruido gaussiano
X_future_d = X_future_d + np.random.normal(0, 0.3, X_future_d.shape)

# Crear DataFrames
df_future_d = pd.DataFrame(X_future_d, columns=feature_names)
df_future_d["target"] = y_future_d
df_future_d["prediction"] = gb_model.predict(X_future_d)

# Generar reporte
report_d = Report(metrics=[DataDriftPreset()])
report_d.run(reference_data=df_train_ref, current_data=df_future_d)

# Guardar HTML
report_d_path = "/app/reports/practica_escenario_d_feature_drift.html"
report_d.save_html(report_d_path)

print(f"✅ Reporte guardado: {report_d_path}")
print(f"📊 Cambios: 10% outliers + Ruido gaussiano (std=0.3)")
print(f"🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!")



ESCENARIO D: FEATURE DRIFT (outliers y ruido)

✅ Reporte guardado: /app/reports/practica_escenario_d_feature_drift.html
📊 Cambios: 10% outliers + Ruido gaussiano (std=0.3)
🔍 ¿Hay Data Drift? ¡Identifícalo en el reporte!


---

## 🎓 TAREA PARA EL ALUMNO

### 📋 Analiza los 4 reportes HTML generados:
1. `practica_escenario_a_sin_cambios.html`
2. `practica_escenario_b_covariate_shift.html`
3. `practica_escenario_c_label_shift.html`
4. `practica_escenario_d_feature_drift.html`

### ❓ Preguntas:
1. **¿Cuál(es) escenario(s) tienen Data Drift detectado?**
2. **¿Qué columnas muestran drift en cada caso?**
3. **¿Qué test estadístico detectó el drift? (chi-square, K-S, etc.)**
4. **¿Cómo cambió la distribución en cada escenario?**

### 💡 Pistas:
- Mira el **"Share of Drifted Columns"** en cada reporte
- Observa los **histogramas** de reference vs current
- Revisa los **p-values**: valores cercanos a 0 = DRIFT detectado
- Compara las **distribuciones de features** entre escenarios


## 📍 Acceso a los reportes


In [16]:
print("\n" + "="*60)
print("RESUMEN DE REPORTES GENERADOS")
print("="*60)

reports = [
    ("Escenario A: Sin cambios", report_a_path),
    ("Escenario B: Covariate Shift", report_b_path),
    ("Escenario C: Label Shift", report_c_path),
    ("Escenario D: Feature Drift", report_d_path),
]

print("\n📁 Reportes guardados en /app/reports/:")
for i, (nombre, ruta) in enumerate(reports, 1):
    print(f"\n  {i}. {nombre}")
    print(f"     Archivo: {ruta}")
    print(f"     ✅ Abre este HTML en el navegador para analizar")

print("\n\n🎯 INSTRUCCIONES:")
print("   1. Abre Jupyter en tu navegador")
print("   2. Descarga cada archivo HTML del directorio /app/reports/")
print("   3. O accede directamente: http://localhost:8888/files/reports/")
print("   4. Analiza cada reporte sin leer el nombre del archivo")
print("   5. Identifica qué escenarios tienen drift")

print("\n" + "="*60)



RESUMEN DE REPORTES GENERADOS

📁 Reportes guardados en /app/reports/:

  1. Escenario A: Sin cambios
     Archivo: /app/reports/practica_escenario_a_sin_cambios.html
     ✅ Abre este HTML en el navegador para analizar

  2. Escenario B: Covariate Shift
     Archivo: /app/reports/practica_escenario_b_covariate_shift.html
     ✅ Abre este HTML en el navegador para analizar

  3. Escenario C: Label Shift
     Archivo: /app/reports/practica_escenario_c_label_shift.html
     ✅ Abre este HTML en el navegador para analizar

  4. Escenario D: Feature Drift
     Archivo: /app/reports/practica_escenario_d_feature_drift.html
     ✅ Abre este HTML en el navegador para analizar


🎯 INSTRUCCIONES:
   1. Abre Jupyter en tu navegador
   2. Descarga cada archivo HTML del directorio /app/reports/
   3. O accede directamente: http://localhost:8888/files/reports/
   4. Analiza cada reporte sin leer el nombre del archivo
   5. Identifica qué escenarios tienen drift

